# Authentication Mechanism 2

CEDA data - via ceda account, http using token


In [1]:
%load_ext autoreload
%autoreload 2

curl --location --request POST 'https://services.ceda.ac.uk/api/token/create/' --header "Authorization: Basic $(echo -n "...:..." | base64)"


In [10]:
import getpass
import json
import requests

from base64 import b64encode
from pathlib import Path
from urllib.parse import urlparse


In [21]:
CEDA_USERNAME = input("Enter CEDA username: ")
CEDA_PASSWORD = getpass.getpass("Enter CEDA password: ")

if not CEDA_USERNAME or not CEDA_PASSWORD:
    raise ValueError("CEDA_USERNAME and/or CEDA_PASSWORD were not provided.")

In [22]:
url = "https://services.ceda.ac.uk/api/token/create/"

username = CEDA_USERNAME
password = CEDA_PASSWORD
token = b64encode(f"{username}:{password}".encode("utf-8")).decode("ascii")
headers = {
    "Authorization": f"Basic {token}",
}

response = requests.request("POST", url, headers=headers)

# If successful, this will return a JSON response containing the token
response_data = json.loads(response.text)
print(response.text)
if response.status_code == 200:
    token = response_data["access_token"]

{"error":"invalid_grant","error_description":"Invalid user credentials"}


In [5]:
data_url = (
    "https://dap.ceda.ac.uk/badc/csip/data/salford-radiometer-1/2005/06/salford-radiometer-1_faccombe_20050624_iwv.nc"
)

output_dir = Path("../data/download")
output_dir.mkdir(parents=True, exist_ok=True)

filename = Path(urlparse(data_url).path).name
output_file = output_dir / filename

with requests.get(data_url, headers=headers, stream=True, timeout=30) as r:
    r.raise_for_status()  # <-- fail fast on 403/404/etc.

    with open(output_file, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

print(f"Downloaded → {output_file}")

requests.get(data_url, headers={"Authorization": f"Bearer {token}"}, stream=True, timeout=30)

Downloaded → ../data/download/salford-radiometer-1_faccombe_20050624_iwv.nc


<Response [200]>